# Importación de los datos
Importación del dataset y lectura de sus características

In [ ]:
import pandas as pd

DATOS = pd.read_csv("housing.csv")

### Descripción de las variables
1. **longitude:** A measure of how far west a house is; a higher value is farther west

2. **latitude:** A measure of how far north a house is; a higher value is farther north

3. **housingMedianAge:** Median age of a house within a block; a lower number is a newer building

4. **totalRooms:** Total number of rooms within a block

5. **totalBedrooms:** Total number of bedrooms within a block

6. **population:** Total number of people residing within a block

7. **households:** Total number of households (hogares, viviendas, domicilios), a group of people residing within a home unit, for a block

8. **medianIncome:** Median income for households within a block of houses (measured in tens of thousands of US Dollars)

9. **medianHouseValue:** Median house value for households within a block (measured in US Dollars)

10. **oceanProximity:** Location of the house w.r.t ocean/sea

In [ ]:
DATOS.head()

In [ ]:
DATOS.describe()

In [ ]:
DATOS.info()

In [ ]:
DATOS.isnull().sum()

In [ ]:
# 2 opciones: Eliminar los valores nulos o replazarlos con la media
data_non_null = DATOS.dropna()
data_non_null.info()

data_clean = DATOS.copy()
data_clean["total_bedrooms"] = data_clean["total_bedrooms"].fillna(value=data_clean["total_bedrooms"].mean())

#  Imprimimos para ver si la media cambia:
# Parece que 'total_bedrooms' no cambia (Si cambias los datos nulos por la media no alteras la media global)
# Pero otros valores si pueden cambiar (Si eliminas datos con 'total_bedrooms' nulos, realmente estás borrando
# esa fila de datos entera, por lo que eliminas ese valor del conteo de la media)
print("\ntotal_bedrooms:")
print(f"Media sin valores nulos:  {data_non_null["total_bedrooms"].mean()}")
print(f"Media imputando la media: {data_clean["total_bedrooms"].mean()}")

print("\nhousing_median_age:")
print(f"Media sin valores nulos:  {data_non_null["housing_median_age"].mean()}")
print(f"Media imputando la media: {data_clean["housing_median_age"].mean()}")

In [ ]:
# Elijo quedarme con el dataframe sin datos nulos (y sin imputación de datos)
data = data_non_null.copy()

# Visualización de los datos (antes del procesamiento)
Mediante matplotlib y pandas

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# --- Dibujamos varios gráficos ---

def plotGraphs(data):
    """
    Crea 4 gráficos para el conjunto de datos especificado. No muestra datos muy relevantes, simplemente prueba a mostrar datos en diferentes plots
    """

    plt.style.use('_mpl-gallery')
    # Creamos los ejes de la figura
    fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(nrows=2, ncols=2, figsize=(10, 6))
    """
    nrows, ncols: Número de subplots en cada línea/columna 
    """

    # Dibujamos cada gráfico
    ax1.set_title("median_house_value")
    ax1.hist(data["median_house_value"], bins=50, linewidth=0.5, edgecolor="white")

    ax2.set_title("latitude vs longitude")
    ax2.scatter(np.array(data["latitude"]), np.array(data["longitude"]), color="green")

    ax3.set_title("median_house_value vs. total_rooms")
    ax3.scatter(
        np.array(data["median_house_value"]),
        np.array(data["total_rooms"])
    )

    ax4.set_title("ocean_proximity")
    x_categorica = data["ocean_proximity"].unique()
    y_categorica = np.array(data["ocean_proximity"].value_counts())
    ax4.bar(x_categorica, y_categorica, linewidth=0.5, color="blue")

    plt.show()

In [ ]:
plotGraphs(data)

In [ ]:
# Para ver boxplots de cada serie de datos numéricos
def plotBoxplots(data, figsize=(16, 6)):
    """
    Muestra un boxplot para cada valor numérico de 'data'
    """

    # Creamos los ejes de la figura
    plot_data = data.select_dtypes(include="number")
    rows = 2
    cols = (len(plot_data.columns) + 1) // rows

    fig, axs = plt.subplots(nrows=rows, ncols=cols, figsize=figsize, layout="constrained")
    # Aplanamos la matriz para recorrer cada uno de sus valores (no nos interesan sus índices x,y)
    axs = axs.flatten()

    for i, col in enumerate(plot_data.columns):
        ax = axs[i]
        ax.set_title(col)
        data = np.array(plot_data[col])
        ax.boxplot(data, showmeans=True, showfliers=True)

    # Ocultamos las celdas vacías (si las hay)
    for j in range(i + 1, len(axs)):
        axs[j].set_visible(False)

    plt.show()


# Para ver bar plots de cada serie de datos numéricos
def plotBarPlots(data, figsize=(16, 6)):
    """
    Muestra un boxplot para cada valor numérico de 'data'
    """

    # Creamos los ejes de la figura
    plot_data = data.select_dtypes(include="number")
    rows = 2
    cols = (len(plot_data.columns) + 1) // rows

    fig, axs = plt.subplots(
        nrows=rows, ncols=cols, figsize=figsize, layout="constrained"
    )
    # Aplanamos la matriz para recorrer cada uno de sus valores (no nos interesan sus índices x,y)
    axs = axs.flatten()

    for i, col in enumerate(plot_data.columns):
        ax = axs[i]
        ax.set_title(col)
        data = np.array(plot_data[col])
        ax.hist(data, bins=50, edgecolor="white")

    # Ocultamos las celdas vacías (si las hay)
    for j in range(i + 1, len(axs)):
        axs[j].set_visible(False)

    plt.show()

In [ ]:
plotBoxplots(data)
# plotBarPlots(data)

# Eliminación de outliers de todas las variables
Primero, creamos nuevas columnas con las variables relativas a 'household'. Por ejemplo, en un bloque de casas con 500 habitaciones es importante conocer si hay 500 residentes/habitantes o 1000

Después, para cada variable numérica, miramos cada uno de sus valores y eliminamos los que estén por muy alejados de la media (1.5 * RangoIntercuartil)

### Creación de columnas con valores relativos
Las columnas 'total_rooms', 'total_bedrooms' y 'population' dependen del número total de 'households' (apartamentos, viviendas) de cada bloque

In [ ]:
# 'total_rooms', 'total_bedrooms' y 'population' relativo a 'households'
data["rooms_per_household"] = data["total_rooms"] / data["households"]
data["bedrooms_per_household"] = data["total_bedrooms"] / data["households"]
data["population_per_household"] = data["population"] / data["households"]
data.head()

In [ ]:
# Ahora ya podemos ver mejor las variables
plotBarPlots(data)
plotBoxplots(data)

In [36]:
# Imprimimos por pantalla varios valores para ver como se distribuyen los datos
column = data["rooms_per_household"]
print(
    column.min(),
    column.median(),
    column.mean(),
    column.max(),
)
print(column.std())
# Columna de 'rooms_per_household' ordenada en orden descendente
print(
    data.sort_values("rooms_per_household", ascending=False).loc[
        :, "rooms_per_household"
    ]
)

2.1339285714285716 5.147133472678412 5.151521531470408 8.320441988950277
1.020861060244764
2219     8.320442
10248    8.307054
20360    8.287611
365      8.280000
1550     8.273632
           ...   
14039    2.155556
4599     2.153094
17553    2.146739
4820     2.140086
5597     2.133929
Name: rooms_per_household, Length: 14558, dtype: float64


### Eliminación de datos fuera de rango
Primero, quitamos las casas con 'median_house_value' igual a 500001 y las casas con 'housing_median_age' iguales a 52 (son valores mal etiquetados, están aproximados a ese valor como 'valor máximo')

Luego, eliminamos outliers de las variables relativas

In [ ]:
# -- Quitamos las casas con 'median_house_value' igual a 500001 y las casas con 'median_house_age' iguales a 52
data = data[data.median_house_value < 500001]
data = data[data.housing_median_age < 52]
plotBarPlots(data)

In [ ]:
# --- Eliminación de outliers ---
# A partir de las nuevas variables relativas, se eliminan los valores muy alejados de la media

separacion_maxima = 1.5 # 2.5 también sirve
data_sin_outliers = data.copy()

# Solo nos interesan las variables numéricas, no categóricas
variables_numericas = data_sin_outliers.select_dtypes(include="number")
print(list(variables_numericas.columns))

print(" COLUMN NAME  |  Q1  |  Q3  |  RI")
for col in variables_numericas.columns:
    columnName = variables_numericas[col]
    # print("\n", col, "\n", column)

    # IQR o RI: Rango intercuartil
    Q1, Q3 = variables_numericas[col].quantile([0.25, 0.75])
    RI = Q3 - Q1
    print(col, Q1, Q3, RI)

    # Eliminamos los outliers por encima del valor especificado
    """ data_limpio = data[(data["columna"] >= Q1 - 1.5 * RI) & (data["columna"] <= Q3 + 1.5 * RI)] """
    data_sin_outliers = data_sin_outliers[
        (data_sin_outliers[col] >= Q1 - separacion_maxima * RI)
        & (data_sin_outliers[col] <= Q3 + separacion_maxima * RI)
    ]

In [ ]:
# Volvemos a visualizar los datos
plotBarPlots(data, figsize=(16, 4))
plotBoxplots(data, figsize=(16, 4))
print("\nDATOS SIN OUTLIERS")
plotBarPlots(data_sin_outliers, figsize=(16, 4))
plotBoxplots(data_sin_outliers, figsize=(16, 4))

In [ ]:
# Volvemos a guardar los nuevos datos como 'data'
data = data_sin_outliers.copy()
data.describe()

# Normalización de los datos que queremos predecir

Normalización de datos: Asegura que los rangos de datos están correctamente escalados. Se deben visualizar los datos antes y después para identificar el método más apropiado

- Normalización Min-Max: Su salida está dentro del intervalo [0, 1]. Uso: datos distribuidos uniformemente

- Estandarización (o Z-score): Transforma los datos para que tengan media 0 y desviación estándar 1. Uso: datos distribuidos normalmente o con valores atípicos

Parece que siguen una distribución normal, así que se pueden estandarizar con Z-score

In [ ]:
# Estandarización

# Notas aparte

In [ ]:
# NOTAS APARTE
# house = data.iloc[i, :] # Para mirar casa por casa
# house